### **VLM Server to run the baseline VLMs**

**(It was deployed and used on Kaggle)**

In [1]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [91.2 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     


In [2]:
!pkill ollama
import os
import threading
import subprocess

def run_ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.run(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=run_ollama).start()

In [3]:
import requests
import json
from tqdm import tqdm

def pull_model_with_progress(model_name):
    print(f"Starting download for {model_name}...")
    url = "http://localhost:11434/api/pull"
    payload = {"model": model_name}
    
    response = requests.post(url, json=payload, stream=True)
    response.raise_for_status()

    pbars = {}
    for line in response.iter_lines():
        if line:
            data = json.loads(line)
            digest = data.get("digest")
            total = data.get("total")
            completed = data.get("completed")

            if digest:
                label = digest[:12]
                if digest not in pbars and total:
                    pbars[digest] = tqdm(total=total, unit='B', unit_scale=True, desc=f"Pulling {label}", leave=False)
                if digest in pbars and completed:
                    pbars[digest].n = completed
                    pbars[digest].refresh()

    for pb in pbars.values():
        pb.close()
    print(f"\nSuccessfully pulled {model_name}")

pull_model_with_progress("medgemma:27b")

Starting download for medgemma:27b...


Pulling sha256:4b97d: 100%|██████████| 17.4G/17.4G [01:06<00:00, 262MB/s]   
Pulling sha256:f79ea:   0%|          | 0.00/394 [00:00<?, ?B/s]
Pulling sha256:f79ea: 100%|██████████| 394/394 [00:01<00:00, 387B/s]

Pulling sha256:12cef:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 66.3kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 49.3kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 39.4kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 32.7kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 28.0kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 24.5kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 21.7kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 19.6kB/s]

Pulling sha256:12cef: 100%|██████████| 11.7k/11.7k [00:00<00:00, 17.8kB/s]

Pulling sha256:12cef: 100%|██████████|


Successfully pulled medgemma:27b


In [4]:
from pyngrok import ngrok
import getpass

# 1. Enter your ngrok authtoken
conf_token = "2KaciyDaZVzw8pGV53XiXSriWRz_2brbTmELYi5Qi4Wksj582"
ngrok.set_auth_token(conf_token)

# 2. Kill old tunnels and open a new one to port 11434
ngrok.kill()
try:
    public_url = ngrok.connect(11434).public_url
    print(f"\nPublic URL: {public_url}")
except Exception as e:
    print(f"Error connecting: {e}")

                                                                                                    
Public URL: https://6c61-34-11-232-230.ngrok-free.app


In [5]:
!curl http://localhost:11434/api/tags

{"models":[{"name":"medgemma:27b","model":"medgemma:27b","modified_at":"2026-05-09T02:21:57.740449186Z","size":17397224125,"digest":"58238ae38f99827496301de22016b2f94157552971f5a38db226f13802c2437e","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"27.4B","quantization_level":"Q4_K_M"}}]}